In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        if len(documents) == 0:
            return

        texts = [document.text for document in documents]

        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(
        self,
        query: str,
        top_k: int = 5,
        metadata_filter: dict[str, str] | None = None
    ) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        if metadata_filter is None:
            candidate_indices = list(range(len(self.documents)))
        else:
            candidate_indices = []

            for index, document in enumerate(self.documents):
                matches_filter = True

                for key, value in metadata_filter.items():
                    if key not in document.metadata:
                        matches_filter = False
                        break

                    if str(document.metadata[key]) != str(value):
                        matches_filter = False
                        break

                if matches_filter:
                    candidate_indices.append(index)

        if len(candidate_indices) == 0:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        candidate_embeddings = self.embeddings[candidate_indices]
        scores = candidate_embeddings @ query_embedding

        sorted_positions = np.argsort(scores)[::-1][:top_k]

        results = []

        for position in sorted_positions:
            original_index = candidate_indices[position]

            result = SearchResult(
                score=float(scores[position]),
                document=self.documents[original_index]
            )

            results.append(result)

        return results

In [4]:
dataset_path = Path("news+aggregator") / "newsCorpora.csv"

if not dataset_path.exists():
    raise FileNotFoundError("No se encontró newsCorpora.csv dentro de la carpeta news+aggregator.")

columns = [
    "id",
    "title",
    "url",
    "publisher",
    "category",
    "story",
    "hostname",
    "timestamp"
]

df_news = pd.read_csv(
    dataset_path,
    sep="\t",
    header=None,
    names=columns,
    encoding="utf-8",
    on_bad_lines="skip"
)

df_news = df_news.fillna("")

df_news.head()

,id,title,url,publisher,category,story,hostname,timestamp
0,1,"Fed official says weak data caused by weather,...",http://www.latimes.com/business/money/la-fi-mo...,Los Angeles Times,b,ddUyU0VZz0BRneMioxUPQVP6sIxvM,www.latimes.com,1394470370698
1,2,Fed's Charles Plosser sees high bar for change...,http://www.livemint.com/Politics/H2EvwJSK2VE6O...,Livemint,b,ddUyU0VZz0BRneMioxUPQVP6sIxvM,www.livemint.com,1394470371207
2,3,US open: Stocks fall after Fed official hints ...,http://www.ifamagazine.com/news/us-open-stocks...,IFA Magazine,b,ddUyU0VZz0BRneMioxUPQVP6sIxvM,www.ifamagazine.com,1394470371550
3,4,"Fed risks falling 'behind the curve', Charles ...",http://www.ifamagazine.com/news/fed-risks-fall...,IFA Magazine,b,ddUyU0VZz0BRneMioxUPQVP6sIxvM,www.ifamagazine.com,1394470371793
4,5,Fed's Plosser: Nasty Weather Has Curbed Job Gr...,http://www.moneynews.com/Economy/federal-reser...,Moneynews,b,ddUyU0VZz0BRneMioxUPQVP6sIxvM,www.moneynews.com,1394470372027


In [5]:
df_news["category"].value_counts()

category
e    152469
b    115967
t    108344
m     45639
Name: count, dtype: int64

In [6]:
sample_size = 5000

if len(df_news) > sample_size:
    df_news_sample = df_news.sample(sample_size, random_state=42)
else:
    df_news_sample = df_news.copy()

df_news_sample = df_news_sample.reset_index(drop=True)

print(f"Noticias usadas: {len(df_news_sample)}")

Noticias usadas: 5000


In [7]:
news_documents = []

for _, row in df_news_sample.iterrows():
    text = str(row["title"])

    metadata = {
        "id": str(row["id"]),
        "url": str(row["url"]),
        "publisher": str(row["publisher"]),
        "category": str(row["category"]),
        "story": str(row["story"]),
        "hostname": str(row["hostname"]),
        "timestamp": str(row["timestamp"])
    }

    document = Document(text=text, metadata=metadata)
    news_documents.append(document)

print(f"Documentos creados: {len(news_documents)}")

Documentos creados: 5000


In [8]:
filtered_vector_store = FilteredVectorStore(embedding_model)

filtered_vector_store.add_documents(news_documents)

print(f"Documentos agregados al FilteredVectorStore: {len(filtered_vector_store.documents)}")
print(f"Forma de los embeddings: {filtered_vector_store.embeddings.shape}")

Documentos agregados al FilteredVectorStore: 5000
Forma de los embeddings: (5000, 384)


In [9]:
def mostrar_resultados_filtrados(query: str, metadata_filter: dict[str, str], results: list[SearchResult]):
    print("=" * 100)
    print(f"Consulta: {query}")
    print(f"Filtro: {metadata_filter}")
    print("=" * 100)

    if len(results) == 0:
        print("No se encontraron resultados.")
        return

    for i, result in enumerate(results, start=1):
        print(f"\nResultado {i}")
        print(f"Score: {result.score:.4f}")
        print(f"Texto: {result.document.text}")
        print("Metadatos:")

        for key, value in result.document.metadata.items():
            print(f"  {key}: {value}")

In [10]:
filtered_queries = [
    {
        "query": "technology companies and new software products",
        "filter": {"category": "t"}
    },
    {
        "query": "movies actors music and celebrities",
        "filter": {"category": "e"}
    },
    {
        "query": "medical research diseases and health problems",
        "filter": {"category": "m"}
    },
    {
        "query": "stock market companies and economic growth",
        "filter": {"category": "b"}
    },
    {
        "query": "smartphones internet apps and devices",
        "filter": {"category": "t"}
    }
]

for item in filtered_queries:
    results = filtered_vector_store.search(
        query=item["query"],
        top_k=5,
        metadata_filter=item["filter"]
    )

    mostrar_resultados_filtrados(
        query=item["query"],
        metadata_filter=item["filter"],
        results=results
    )

Consulta: technology companies and new software products
Filtro: {'category': 't'}

Resultado 1
Score: 0.5191
Texto: Sector Update: Tech Technology
Metadatos:
  id: 303180
  url: http://www.nasdaq.com/article/sector-update-tech-technology6-cm362674
  publisher: NASDAQ
  category: t
  story: dyqLZK_KX89ISTMh4521s5XG6ckKM
  hostname: www.nasdaq.com
  timestamp: 1403128131141

Resultado 2
Score: 0.5041
Texto: Microsoft Corporation (MSFT) news: Microsoft Will Prove To Be A Good Investment
Metadatos:
  id: 13733
  url: http://seekingalpha.com/article/2091933-microsoft-will-prove-to-be-a-good-investment\?source=google_news
  publisher: Seeking Alpha
  category: t
  story: dWMNxDAooBOYngM9Rs86MyAdB9x7M
  hostname: seekingalpha.com
  timestamp: 1395060612281

Resultado 3
Score: 0.4967
Texto: Siri software maker Nuance in sale talks: WSJ
Metadatos:
  id: 295222
  url: http://www.globalpost.com/dispatch/news/thomson-reuters/140616/siri-software-maker-nuance-sale-talks-wsj
  publisher: GlobalPost

In [11]:
df_news_sample["publisher"].value_counts().head(10)

publisher
Reuters                            62
Huffington Post                    33
Contactmusic.com                   31
NASDAQ                             31
RTT News                           26
Businessweek                       26
Entertainmentwise                  22
Los Angeles Times                  22
International Business Times UK    22
Examiner.com                       22
Name: count, dtype: int64

In [12]:
results = filtered_vector_store.search(
    query="technology business and companies",
    top_k=5,
    metadata_filter={"publisher": "Reuters"}
)

mostrar_resultados_filtrados(
    query="technology business and companies",
    metadata_filter={"publisher": "Reuters"},
    results=results
)

Consulta: technology business and companies
Filtro: {'publisher': 'Reuters'}

Resultado 1
Score: 0.3926
Texto: GLOBAL MARKETS-Tech stocks sink Wall Street; US bonds rally
Metadatos:
  id: 101126
  url: http://in.reuters.com/article/2014/04/10/markets-global-idINL2N0N21EW20140410
  publisher: Reuters
  category: b
  story: dDtTmiUm0P1qeMMK8D7BMIAgeWToM
  hostname: in.reuters.com
  timestamp: 1397287862758

Resultado 2
Score: 0.3373
Texto: Fiat open to alliances if they boost cost structure, position
Metadatos:
  id: 410764
  url: http://in.reuters.com/article/2014/07/30/fiatchrysler-alliances-idINL6N0Q54MW20140730
  publisher: Reuters
  category: b
  story: dAXEYZyNSNuoMLMgHGueG8gj53ONM
  hostname: in.reuters.com
  timestamp: 1406930660746

Resultado 3
Score: 0.3102
Texto: US STOCKS-Wall St advances; Internet stocks lift Nasdaq
Metadatos:
  id: 223779
  url: http://in.reuters.com/article/2014/05/19/markets-usa-stocks-idINL1N0O51E720140519
  publisher: Reuters
  category: b
  story: dOff